# B2-023-generative-models-diffusion — Practice p23 — Solution

**Type:** challenge · **Difficulty:** advanced · **Concepts:** denoising-diffusion-probabilistic-models

*55 minutes.*  
**Set:** C  
**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

## Supplied cell (copied verbatim from the statement)

In [ ]:
import math
import torch
from torch import nn

X0 = torch.tensor([[2.0, -1.0], [0.0, 1.5]])
T_IDX = torch.tensor([2, 4])
EPS = torch.tensor([[1.0, 1.0], [-0.5, 0.5]])
X_T = torch.tensor([[1.5, -0.5], [-1.0, 1.0]])
NOISES = torch.tensor([[[0.3, -0.3], [0.1, 0.1]],
                       [[1.0, 0.5], [-0.5, 1.0]],
                       [[-0.2, 0.4], [0.6, -0.8]],
                       [[0.5, 0.5], [1.0, -1.0]]])


class ToyEps(nn.Module):
    def forward(self, x_t, t_scaled):
        return 0.25 * x_t - t_scaled


ATOL = 1e-5
RTOL = 1e-5

## Solution

### (A) Running the code as given

The teammate's three functions are copied verbatim below with a `buggy_` prefix, together with this unit's `make_schedule` (p09 contract).

In [ ]:
def make_schedule(T, beta_1, beta_T):
    if not isinstance(T, int) or isinstance(T, bool) or T < 2:
        raise ValueError("T must be an int >= 2")
    if not (0 < beta_1 <= beta_T < 1):
        raise ValueError("need 0 < beta_1 <= beta_T < 1")
    beta = torch.zeros(T + 1, dtype=torch.float64)
    beta[1:] = torch.linspace(beta_1, beta_T, T, dtype=torch.float64)
    alpha = 1 - beta
    alpha_bar = torch.cumprod(alpha, dim=0)
    beta_tilde = torch.zeros(T + 1, dtype=torch.float64)
    beta_tilde[1:] = (1 - alpha_bar[:-1]) / (1 - alpha_bar[1:]) * beta[1:]
    return {"beta": beta, "alpha": alpha, "alpha_bar": alpha_bar, "beta_tilde": beta_tilde}


def buggy_q_sample(x0, t, eps, schedule):
    a = schedule["alpha"][t].to(x0.dtype).unsqueeze(1)
    return a.sqrt() * x0 + (1 - a).sqrt() * eps


def buggy_p_sample(model, x_t, t, z, schedule):
    T = schedule["beta"].shape[0] - 1
    beta, alpha, alpha_bar = (schedule[k][t].item() for k in ("beta", "alpha", "alpha_bar"))
    eps_hat = model(x_t, torch.full((x_t.shape[0], 1), t / T, dtype=x_t.dtype))
    mean = (x_t - beta / math.sqrt(1 - alpha_bar) * eps_hat) / math.sqrt(alpha)
    if t == 1:
        return mean
    sigma = schedule["beta_tilde"][t].item()
    return mean + sigma * z


def buggy_sample_loop(model, x_T, noises, schedule):
    T = schedule["beta"].shape[0] - 1
    x = x_T
    for t in range(T, 1, -1):
        x = buggy_p_sample(model, x, t, noises[t - 1], schedule)
    return x


schedule = make_schedule(4, 0.1, 0.4)
buggy_q = buggy_q_sample(X0, T_IDX, EPS, schedule)
buggy_loop = buggy_sample_loop(ToyEps(), X_T, NOISES, schedule)
print("buggy q_sample   :", [[round(v, 10) for v in row] for row in buggy_q.tolist()])
print("buggy sample_loop:", [[round(v, 10) for v in row] for row in buggy_loop.tolist()])

### (B) The three defects

1. **$\alpha$/$\bar\alpha$ confusion** in `q_sample`: `a = schedule["alpha"][t]`. The closed form of Session 3 is $x_t=\sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\varepsilon$, so the coefficient must be `schedule["alpha_bar"][t]`; with `alpha` the function applies **one** step of noise to $x_0$ instead of $t$ steps.
2. **Wrong sampling $\sigma_t$** (variance used as standard deviation) in `p_sample`: `sigma = schedule["beta_tilde"][t].item()` followed by `mean + sigma * z`. Session 4 pins $\sigma_t^2=\tilde\beta_t$, so the noise must be scaled by the **standard deviation** $\sigma_t=\sqrt{\tilde\beta_t}$: `math.sqrt(schedule["beta_tilde"][t].item()) * z`.
3. **Off-by-one timestep range** in `sample_loop`: `for t in range(T, 1, -1)` visits $t=T,\dots,2$ and stops before $t=1$. Session 4 requires $t=T,T-1,\dots,1$, i.e. `range(T, 0, -1)`, so the final noiseless step from $x_1$ to $x_0$ is applied.

### (C) Repaired implementation (same signatures)

In [ ]:
def q_sample(x0, t, eps, schedule):
    ab = schedule["alpha_bar"][t].to(x0.dtype).unsqueeze(1)
    return ab.sqrt() * x0 + (1 - ab).sqrt() * eps


def p_sample(model, x_t, t, z, schedule):
    T = schedule["beta"].shape[0] - 1
    beta, alpha, alpha_bar = (schedule[k][t].item() for k in ("beta", "alpha", "alpha_bar"))
    eps_hat = model(x_t, torch.full((x_t.shape[0], 1), t / T, dtype=x_t.dtype))
    mean = (x_t - beta / math.sqrt(1 - alpha_bar) * eps_hat) / math.sqrt(alpha)
    if t == 1:
        return mean
    sigma = math.sqrt(schedule["beta_tilde"][t].item())
    return mean + sigma * z


def sample_loop(model, x_T, noises, schedule):
    T = schedule["beta"].shape[0] - 1
    x = x_T
    for t in range(T, 0, -1):
        x = p_sample(model, x, t, noises[t - 1], schedule)
    return x


fixed_q = q_sample(X0, T_IDX, EPS, schedule)
fixed_loop = sample_loop(ToyEps(), X_T, NOISES, schedule)
print("repaired q_sample   :", fixed_q.tolist())
print("repaired sample_loop:", fixed_loop.tolist())

### (D) One defect at a time

In [ ]:
# Defect 1 alone: alpha instead of alpha_bar in q_sample (the buggy q_sample is exactly that single defect).
only_alpha_q = buggy_q_sample(X0, T_IDX, EPS, schedule)


# Defect 2 alone: beta_tilde used as the standard deviation, in an otherwise repaired loop.
def p_sample_variance_as_std(model, x_t, t, z, schedule):
    T = schedule["beta"].shape[0] - 1
    beta, alpha, alpha_bar = (schedule[k][t].item() for k in ("beta", "alpha", "alpha_bar"))
    eps_hat = model(x_t, torch.full((x_t.shape[0], 1), t / T, dtype=x_t.dtype))
    mean = (x_t - beta / math.sqrt(1 - alpha_bar) * eps_hat) / math.sqrt(alpha)
    if t == 1:
        return mean
    return mean + schedule["beta_tilde"][t].item() * z


def loop_with(step, stop):
    def run(model, x_T, noises, schedule):
        T = schedule["beta"].shape[0] - 1
        x = x_T
        for t in range(T, stop, -1):
            x = step(model, x, t, noises[t - 1], schedule)
        return x
    return run


only_sigma_loop = loop_with(p_sample_variance_as_std, 0)(ToyEps(), X_T, NOISES, schedule)
# Defect 3 alone: the loop stops at t = 2, with the repaired p_sample.
only_range_loop = loop_with(p_sample, 1)(ToyEps(), X_T, NOISES, schedule)

diff_alpha = (only_alpha_q - fixed_q).abs().max().item()
diff_sigma = (only_sigma_loop - fixed_loop).abs().max().item()
diff_range = (only_range_loop - fixed_loop).abs().max().item()
print("alpha/alpha_bar only:", only_alpha_q.tolist(), " max diff", round(diff_alpha, 6))
print("sigma only          :", only_sigma_loop.tolist(), " max diff", round(diff_sigma, 6))
print("range only          :", only_range_loop.tolist(), " max diff", round(diff_range, 6))


class Recorder(nn.Module):
    def __init__(self):
        super().__init__()
        self.visited = []

    def forward(self, x_t, t_scaled):
        self.visited.append(t_scaled[0, 0].item())
        return 0.25 * x_t - t_scaled


recorder = Recorder()
sample_loop(recorder, X_T, NOISES, schedule)
XT_probe = torch.tensor([[0.3, -0.7], [1.1, 0.2]])
step1_with_noise = p_sample(ToyEps(), XT_probe, 1, NOISES[0], schedule)
step1_without_noise = p_sample(ToyEps(), XT_probe, 1, torch.zeros_like(NOISES[0]), schedule)
print("visited t/T:", recorder.visited)

### (E) Explanations

**(i)** At $t=1$, $\bar\alpha_1=\alpha_1$ (a product of one factor), so `alpha[1]` and `alpha_bar[1]` are the same number and the buggy and repaired `q_sample` agree exactly; only a probe at some $t\ge2$, where $\bar\alpha_t=\alpha_1\cdots\alpha_t<\alpha_t$, can expose the confusion (the literals use $t=2$ and $t=4$).

**(ii)** $\tilde\beta_t$ is a **variance**, so multiplying $z$ by $\tilde\beta_t$ gives noise of variance $\tilde\beta_t^2$, far too little noise when $\tilde\beta_t<1$ (e.g. $\tilde\beta_4=0.284404$ here is used as the standard deviation instead of $\sqrt{0.284404}=0.533295$); the reverse chain then under-disperses its samples. The legitimate alternative $\sigma_t^2=\beta_t$ is a different, correctly used **variance** choice (noise scaled by $\sqrt{\beta_t}$, a larger standard deviation than $\sqrt{\tilde\beta_t}$); it changes the per-step noise level by design, whereas the defect is a units error that matches neither convention.

**(iii)** The buggy loop stops after $t=2$ and returns $x_1$, not $x_0$: it skips the final noiseless step $x_0=\frac1{\sqrt{\alpha_1}}\big(x_1-\frac{\beta_1}{\sqrt{1-\bar\alpha_1}}\hat\varepsilon\big)$. Its output still "looks like" a sample because $x_1$ is only one small forward step away from $x_0$ ($\bar\alpha_1=\alpha_1=1-\beta_1$ is close to $1$ for a small $\beta_1$), so it has the right overall location and shape, just with a little residual noise and without the last denoising correction — a defect only a probe of the final values or of the visited timesteps reveals.

### Answer check

In [ ]:
# (C) repaired values
assert torch.allclose(fixed_q, torch.tensor([[2.2262065411, -0.3193778992], [-0.4176122546, 1.2424757481]]), atol=ATOL, rtol=RTOL)
assert torch.allclose(fixed_loop, torch.tensor([[3.5601835251, 1.2876268625], [0.7925575972, 1.8815526962]]), atol=ATOL, rtol=RTOL)
# (D) each single-defect variant differs by more than 0.05 (max absolute difference)
assert diff_alpha > 0.05
assert diff_sigma > 0.05
assert diff_range > 0.05
# recording model: t_scaled visited in descending order 1.0, 0.75, 0.5, 0.25
assert recorder.visited == [1.0, 0.75, 0.5, 0.25]
# the repaired t = 1 step ignores its noise argument
assert torch.equal(step1_with_noise, step1_without_noise)
# (A) the buggy values differ from the repaired ones
assert not torch.allclose(buggy_q, fixed_q, atol=1e-08, rtol=1e-05) and not torch.allclose(buggy_loop, fixed_loop, atol=1e-08, rtol=1e-05)
# (E)(ii) the numbers quoted for beta_tilde_4
assert round(schedule["beta_tilde"][4].item(), 6) == 0.284404 and round(math.sqrt(schedule["beta_tilde"][4].item()), 6) == 0.533295
# (E)(i) at t = 1 the alpha/alpha_bar defect is invisible
t1 = torch.tensor([1, 1])
assert torch.equal(buggy_q_sample(X0, t1, EPS, schedule), q_sample(X0, t1, EPS, schedule))